In [1]:
print("Downloading files from public repository...")

# The -f flag tells curl to fail on an error (like 404)
!curl -Lf https://raw.githubusercontent.com/Denis2054/Context-Engineering-for-Multi-Agent-Systems/main/commons/engine/utils.py --output utils.py
!curl -Lf https://raw.githubusercontent.com/Denis2054/Context-Engineering-for-Multi-Agent-Systems/main/commons/engine/helpers.py --output helpers.py
!curl -Lf https://raw.githubusercontent.com/Denis2054/Context-Engineering-for-Multi-Agent-Systems/main/commons/engine/agents.py --output agents.py
!curl -Lf https://raw.githubusercontent.com/Denis2054/Context-Engineering-for-Multi-Agent-Systems/main/commons/engine/registry.py --output registry.py
!curl -Lf https://raw.githubusercontent.com/Denis2054/Context-Engineering-for-Multi-Agent-Systems/main/commons/engine/engine.py --output engine.py
# (You might want to add a check here to see if the files actually exist now)
print("✅ File download attempt finished!")


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  2013  100  2013    0     0  14877      0 --:--:-- --:--:-- --:--:-- 14911
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  6481  100  6481    0     0  48040      0 --:--:-- --:--:-- --:--:-- 48007
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  9440  100  9440    0     0  65576      0 --:--:-- --:--:-- --:--:-- 66013
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  3611  100  3611    0     0  26388      0 --:--:-- --:--:-- --:--:-- 26551
  % Total    % Received % Xferd  Average Speed   Tim

In [2]:
#Installation and Client Setup

# Import the setup functions from your new utility file
import utils

# Run the installation
utils.install_dependencies()

# Initialize the OpenAI and Pinecone clients
client, pc = utils.initialize_clients()

🚀 Installing required packages...
✅ All packages installed successfully.

🔑 Initializing API clients...
   - OpenAI client initialized.
   - Pinecone client initialized.
✅ Clients initialized successfully.


In [3]:

# 1. Import the hardened helper functions (LLM, Embeddings, Pinecone)
import helpers

# 2. Import the specialist agent functions (Librarian, Researcher, Writer)
import agents

# 3. Import the AGENT_TOOLKIT object that knows about all the agents
from registry import AGENT_TOOLKIT

# 4. Import the main context_engine function that orchestrates the entire process
from engine import context_engine

In [4]:
import json
import html
import markdown
from IPython.display import display, HTML

def render_trace_dashboard(trace):
    """
    Generates a high-contrast, high-readability HTML dashboard for the Context Engine Trace.
    Includes explicit token metrics and deep charcoal text for maximum clarity.
    """
    # Define CSS with hard-coded high-contrast colors and heavy font weights
    css = """

    """

    status_class = "status-success" if trace.status == "Success" else "status-failure"

    dashboard_html = f"""
    {css}



                Context Engine Trace
                "{html.escape(trace.goal)}"


                {trace.status}

                    ⏱️ TIME: {trace.duration:.2f}s





            Execution Workflow
    """

    for step in trace.steps:
        # Prepare inputs and outputs
        try:
            resolved_ctx = json.dumps(step['resolved_context'], indent=2)
        except:
            resolved_ctx = str(step.get('resolved_context', 'N/A'))

        output_raw = step.get('output', 'N/A')
        rendered_html = ""

        def render_md(text):
            return markdown.markdown(text) if text else "No content recorded."

        if isinstance(output_raw, dict):
            content_found = False
            for key in ['summary', 'answer_with_sources', 'answer', 'output', 'content']:
                if key in output_raw and isinstance(output_raw[key], str):
                    rendered_html = f'{render_md(output_raw[key])}'
                    content_found = True
                    break
            if not content_found:
                rendered_html = f'{html.escape(json.dumps(output_raw, indent=2))}'
        else:
            rendered_html = f'{render_md(str(output_raw))}'

        # Metrics Extraction
        t_in = step.get('tokens_in', '??')
        t_out = step.get('tokens_out', '??')
        t_saved = step.get('tokens_saved', 0)

        metrics_html = f"""

            📥 IN: {t_in}
            📤 OUT: {t_out}
        """
        if t_saved > 0:
            metrics_html += f'📉 SAVED: {t_saved}'
        metrics_html += ""

        # Build Step Card
        step_html = f"""




                            STEP {step['step']}
                            {step['agent']}

                        {metrics_html}

                    OPEN LOGS



                        Input Context (State)
                        ▶ View Resolved Source Data
                        {html.escape(resolved_ctx)}


                        Agent Output
                        {rendered_html}



        """
        dashboard_html += step_html

    # Final Result Section
    if trace.final_output:
        final_content = trace.final_output
        if isinstance(final_content, dict):
            final_content = final_content.get('summary', final_content.get('content', str(final_content)))

        dashboard_html += f"""

            Final Orchestration Result
            {markdown.markdown(str(final_content))}

        """

    dashboard_html += ""
    display(HTML(dashboard_html))

In [5]:
# === ENGINE ROOM: The Main Execution Function ===
# This function contains all the logic to run the engine.
# We define it here so our final cell can be very simple.

import logging
import pprint
import json  # <--- Added for type handling
from IPython.display import display, Markdown

# === ENGINE ROOM: The Main Execution Function (Visualizer Update) ===

import logging
import pprint
import json
from IPython.display import display, Markdown

def execute_and_display(goal, config, client, pc, moderation_active):
    """
    Runs the context engine with HTML dashboard visualization.
    """
    # --- PRE-FLIGHT MODERATION CHECK (on user input) ---\n
    if moderation_active:
        print("--- [Safety Guardrail] Performing Pre-Flight Moderation Check on Goal ---")
        moderation_report = helpers.helper_moderate_content(text_to_moderate=goal, client=client)

        if moderation_report["flagged"]:
            print("\n🛑 Goal failed pre-flight moderation. Execution halted.")
            pprint.pprint(moderation_report)
            return

    logging.info(f"******** Starting Engine for Goal: '{goal}' **********\n")

    # 1. Run the Context Engine
    result, trace = context_engine(
        goal,
        client=client,
        pc=pc,
        **config
    )

    # --- POST-FLIGHT MODERATION CHECK (on AI output) ---\n
    if result and moderation_active:
        # Flatten result for checking
        text_to_check = str(result)
        if isinstance(result, (dict, list)):
            text_to_check = json.dumps(result)

        moderation_report = helpers.helper_moderate_content(text_to_moderate=text_to_check, client=client)

        if moderation_report["flagged"]:
            print("\n🛑 Generated output failed post-flight moderation and will be redacted.")
            result = "[Content flagged as potentially harmful by moderation policy and has been redacted.]"
            # We also scrub the trace result to prevent display of harmful content
            trace.final_output = result

    # 2. Render the HTML Dashboard
    # This replaces the raw text printouts with the visualizer
    if trace:
        render_trace_dashboard(trace)
    else:
        print("Engine failed to initialize trace.")

In [6]:
# =============================================================================
# 🛠️ HOTFIX: ROBUST PLANNER PATCH
# =============================================================================
# Run this cell to overwrite the default planner with a strict-schema version.
# This fixes the "NoneType object has no attribute 'get'" error.

import engine
import json
import logging
from helpers import call_llm_robust

def planner_robust_patch(goal, capabilities, client, generation_model):
    """
    A patched version of the Planner that enforces strict JSON schema for inputs.
    """
    logging.info("[Planner-Patch] Activated. Generating strict schema plan...")

    # UPGRADE: Explicitly showing the required schema in the prompt
    system_prompt = f"""
You are the strategic core of the Context Engine. Analyze the user's high-level GOAL and create a step-by-step EXECUTION PLAN.

AVAILABLE CAPABILITIES
---
{capabilities}
---
END CAPABILITIES

INSTRUCTIONS:
1. The output MUST be a single JSON object with a "plan" key containing a list of step objects.
2. CRITICAL: Every step object MUST strictly follow this schema:
   {{
      "step": ,
      "agent": "",
      "input": {{
          "": ""
      }}
   }}
   (You MUST wrap the agent arguments inside the "input" object. Do not place them at the root of the step object.)

3. Use Context Chaining: format "
STEP_N_OUTPUT
" for values requiring previous outputs.
"""
    try:
        plan_json_string = call_llm_robust(
            system_prompt,
            goal,
            client=client,
            generation_model=generation_model,
            json_mode=True
        )
        plan_data = json.loads(plan_json_string)

        # Extra safety: Check if 'plan' key exists, if not, assume the list is the plan
        if "plan" not in plan_data and isinstance(plan_data, list):
             return plan_data
        return plan_data["plan"]

    except Exception as e:
        logging.error(f"Planner failed to generate a valid plan. Error: {e}")
        raise e

# Apply the patch to the imported engine module
engine.planner = planner_robust_patch
print("✅ Engine successfully patched with Robust Planner.")

✅ Engine successfully patched with Robust Planner.


In [7]:
# 1. Define all configuration variables for this run in a dictionary
config = {
    "index_name": 'genai-mas-mcp-ch3',
    "generation_model": "gpt-5.1",
    "embedding_model": "text-embedding-3-small",
    "namespace_context": 'ContextLibrary',
    "namespace_knowledge": 'KnowledgeStore'
}

In [8]:
#@title CONTROL DECK: Moderation
# 1. Define a simple, safe goal to test the moderation workflow.
goal = "Summarize the key points of the QuantumDrive"

# 2. Define the standard configuration.
config = {
    "index_name": 'genai-mas-mcp-ch3',
    "generation_model": "gpt-5.1",
    "embedding_model": "text-embedding-3-small",
    "namespace_context": 'ContextLibrary',
    "namespace_knowledge": 'KnowledgeStore'
}

# 3. Call the execution function with moderation explicitly activated.
execute_and_display(goal, config, client, pc, moderation_active=True)

--- [Safety Guardrail] Performing Pre-Flight Moderation Check on Goal ---


ERROR:root:OpenAI API Error in call_llm_robust: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}
ERROR:root:OpenAI API Error in call_llm_robust: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}
ERROR:root:OpenAI API Error in call_llm_robust: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}
ERROR:root:OpenAI API Error in call_llm_robust: Error code: 429 - {'error': {'message

In [9]:
#@title Product Marketing Copy Generation
goal = "Analyze the ChronoTech press release and summarize their core product messaging and value proposition. Please cite your sources."


# 2. Use the standard configuration
config = {
    "index_name": 'genai-mas-mcp-ch3',
    "generation_model": "gpt-5.1", # or your preferred model
    "embedding_model": "text-embedding-3-small",
    "namespace_context": 'ContextLibrary',
    "namespace_knowledge": 'KnowledgeStore'
}

# 3. Call the execution function
execute_and_display(goal, config, client, pc,moderation_active=False)

ERROR:root:OpenAI API Error in call_llm_robust: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}
ERROR:root:OpenAI API Error in call_llm_robust: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}
ERROR:root:OpenAI API Error in call_llm_robust: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}
ERROR:root:OpenAI API Error in call_llm_robust: Error code: 429 - {'error': {'message

In [10]:

#@title Writing a brand pitch recommendation

# 1. Define the Goal: A creative or factual task that is deliberately
#    outside the scope of the documents in the knowledge base.
#    - DOMAIN: Universal test applicable to any curated knowledge base.
#    - KEY CAPABILITY: Tests the `Researcher` agent's ability to report a
#      negative finding and the `Writer` agent's ability to handle it gracefully,
#      preventing hallucination.
# goal = "[INSERT YOUR OUT-OF-SCOPE GOAL HERE]"


# =CONTROL DECK 3: A persuasive pitch ===
goal = "Write a persuasive pitch on our brand tone and voice guide"


# 2. Use the same configuration dictionary
config = {
    "index_name": 'genai-mas-mcp-ch3',
    "generation_model": "gpt-5.1", # or your preferred model
    "embedding_model": "text-embedding-3-small",
    "namespace_context": 'ContextLibrary',
    "namespace_knowledge": 'KnowledgeStore'
}

# 3. Call the execution function
execute_and_display(goal, config, client, pc,moderation_active=False)


ERROR:root:OpenAI API Error in call_llm_robust: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}
ERROR:root:OpenAI API Error in call_llm_robust: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}
ERROR:root:OpenAI API Error in call_llm_robust: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}
ERROR:root:OpenAI API Error in call_llm_robust: Error code: 429 - {'error': {'message

In [11]:
#@title CONTROL DECK: Moderation
# 1. Define a simple, safe goal to test the moderation workflow.

# November 20, 2025 goal update:
# The former goal contained the the term "summarize" creating ambiguity.
#goal = "Summarize the key points of the Non-Disclosure Agreement."

# The new goal now explains that the task is not to directly summarize but first retrieve data thus clarifying the ambiguity
goal = "First, retrieve the content of the Non-Disclosure Agreement (NDA) from the knowledge base. Then, summarize its key points."
# The system will then function correctly, sanitize the information and continue the process

# 2. Define the standard configuration.
config = {
    "index_name": 'genai-mas-mcp-ch3',
    "generation_model": "gpt-5.1",
    "embedding_model": "text-embedding-3-small",
    "namespace_context": 'ContextLibrary',
    "namespace_knowledge": 'KnowledgeStore'
}

# 3. Call the execution function with moderation explicitly activated.
execute_and_display(goal, config, client, pc, moderation_active=True)


--- [Safety Guardrail] Performing Pre-Flight Moderation Check on Goal ---


ERROR:root:OpenAI API Error in call_llm_robust: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}
ERROR:root:OpenAI API Error in call_llm_robust: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}
ERROR:root:OpenAI API Error in call_llm_robust: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}
ERROR:root:OpenAI API Error in call_llm_robust: Error code: 429 - {'error': {'message

In [12]:
#@title CONTROL DECK TEMPLATE 1: High-Fidelity RAG

# 1. Define the Goal: A research query that requires a verifiable, cited answer.
#    - DOMAIN: Any knowledge-intensive field (e.g., legal, medical, financial).
#    - KEY CAPABILITY: Tests the high-fidelity `Researcher` agent and its ability
#      to retrieve text with `source` metadata and generate citations.
# goal = "[INSERT YOUR HIGH-FIDELITY RESEARCH GOAL HERE]"

# === CONTROL DECK 1: High-Fidelity RAG in a Legal Context ===
goal = "What are the key confidentiality obligations in the Service Agreement v1, and what is the termination notice period? Please cite your sources."

# === CONTROL DECK 1 (LIMIT TEST): Sanitization of Legal Testimony ===
#goal = "What did Mr. Smith advise his client regarding the assets?"

# 2. Use the standard configuration
config = {
    "index_name": 'genai-mas-mcp-ch3',
    "generation_model": "gpt-5.1", # or your preferred model
    "embedding_model": "text-embedding-3-small",
    "namespace_context": 'ContextLibrary',
    "namespace_knowledge": 'KnowledgeStore'
}

# 3. Call the execution function
execute_and_display(goal, config, client, pc,moderation_active=False)


ERROR:root:OpenAI API Error in call_llm_robust: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}
ERROR:root:OpenAI API Error in call_llm_robust: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}
ERROR:root:OpenAI API Error in call_llm_robust: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}
ERROR:root:OpenAI API Error in call_llm_robust: Error code: 429 - {'error': {'message